# Exploratory Data Analysis (EDA) — Peta Digital UMKM Kuliner Surabaya Timur
### Analisis Multimodal: Tabular, Teks Ulasan, Citra Indikator (YOLOv8), dan Dokumen Regulasi

**Deskripsi Proyek:**
- **Mata Kuliah:** Big Data Analytics
- **Fokus Wilayah:** Surabaya Timur (Sentra Wisata Kuliner & Sekitarnya)
- **Penyimpanan Utama:** Google Drive (Sesuai Arsitektur & Flowchart BDA)
- **Status Output:** Tersinkronisasi Otomatis ke Google Drive

---
## 1. Pendahuluan & Ringkasan Dataset
Notebook ini memuat analisis eksplorasi data (EDA) komprehensif pada **4 modalitas data**:
1. **Data Tabular:** Metadata profil 190 UMKM kuliner di 6 kecamatan Surabaya Timur + stan sentra kuliner.
2. **Data Teks:** 730 ulasan pengunjung dari Google Places API setelah disanitasi & dinormalisasi.
3. **Data Citra/Foto:** Foto hasil kurasi detektor YOLOv8 yang **hanya memuat stan dan indikator target klasifikasi** (QRIS, EDC, POS, Merchant Online) serta telah dianonymisasi (OpenCV face blurring).
4. **Data Dokumen:** Dokumen regulasi/perizinan pendukung penelitian.


In [ ]:
# 1. Inisialisasi Environment & Koneksi Otomatis Google Drive
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Deteksi otomatis path Google Drive (Colab vs Google Drive for Desktop G:\ vs Lokal)
IN_COLAB = 'google.colab' in sys.modules
GDRIVE_DESKTOP_PATH = Path('G:/My Drive')
TARGET_STAGE = 'Iqbal-EDA'

if IN_COLAB:
    print('-> Terdeteksi berjalan di Google Colab. Menghubungkan Google Drive...')
    from google.colab import drive
    drive.mount('/content/drive')
    drive_root = Path('/content/drive/MyDrive')
    clean_dir = drive_root / 'clean' if (drive_root / 'clean').exists() else Path('/content/data/clean')
    OUTPUT_DIR = drive_root / TARGET_STAGE
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    print(f'-> [SUKSES COLAB] Output otomatis tersimpan di Google Drive: {OUTPUT_DIR}')
elif GDRIVE_DESKTOP_PATH.exists():
    print('-> Terdeteksi Google Drive for Desktop (Drive G:).')
    clean_dir = GDRIVE_DESKTOP_PATH / 'clean' if (GDRIVE_DESKTOP_PATH / 'clean').exists() else Path('data/clean')
    OUTPUT_DIR = GDRIVE_DESKTOP_PATH / TARGET_STAGE
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    print(f'-> [SUKSES GDRIVE DESKTOP] Output otomatis tersimpan & ter-sync ke Google Drive: {OUTPUT_DIR}')
else:
    print('-> Menjalankan di penyimpanan lokal.')
    clean_dir = Path('data/clean')
    OUTPUT_DIR = Path('reports') / TARGET_STAGE.lower()
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    print(f'-> Output lokal tersimpan di: {OUTPUT_DIR}')

# Konfigurasi Tampilan Grafik
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120
print(f'Path Dataset Clean : {clean_dir}')
print(f'Path Output Target : {OUTPUT_DIR}')


---
## 2. Pemuatan Data dari Zona Clean (Format Parquet di Google Drive)
Data yang dianalisis berasal langsung dari zona `clean/` di Google Drive yang telah divalidasi, dideduplikasi, dan disaring menggunakan Ultralytics YOLOv8.


In [ ]:
# 2. Load Dataset Bersih Parquet
df_umkm = pd.read_parquet(clean_dir / 'umkm.parquet')
df_ulasan = pd.read_parquet(clean_dir / 'ulasan.parquet')
df_foto = pd.read_parquet(clean_dir / 'foto.parquet')
df_dokumen = pd.read_parquet(clean_dir / 'dokumen.parquet')

print(f'Total UMKM Terdaftar   : {len(df_umkm)}')
print(f'Total Ulasan Bersih    : {len(df_ulasan)}')
print(f'Total Foto Terkurasi   : {len(df_foto)} (Hanya memuat Stan & Indikator Target)')
print(f'Total Dokumen Perizinan: {len(df_dokumen)}')


---
## 3. EDA Modalitas 1 — Data Tabular UMKM Kuliner
Menganalisis sebaran jumlah UMKM per kecamatan di Surabaya Timur, distribusi rating publik, dan proporsi Sentra Wisata Kuliner (SWK).


In [ ]:
# 3.1 Ringkasan Struktur Data UMKM
print('Struktur Kolom df_umkm:')
df_umkm.info()
display(df_umkm.head(5))


In [ ]:
# 3.2 Visualisasi Sebaran UMKM per Kecamatan
plt.figure(figsize=(10, 5))
kec_counts = df_umkm['kecamatan'].value_counts()
sns.barplot(x=kec_counts.values, y=kec_counts.index, palette='viridis')
plt.title('Distribusi Titik UMKM Kuliner per Kecamatan di Surabaya Timur', fontsize=14, fontweight='bold')
plt.xlabel('Jumlah UMKM', fontsize=12)
plt.ylabel('Kecamatan', fontsize=12)
for i, v in enumerate(kec_counts.values):
    plt.text(v + 0.5, i, str(v), color='black', va='center', fontweight='bold')
plt.tight_layout()
# Otomatis simpan langsung ke Google Drive
save_path = OUTPUT_DIR / '01_distribusi_umkm_kecamatan.png'
plt.savefig(save_path, dpi=300, bbox_inches='tight')
print(f'-> [TERSIMPAN DI GDRIVE] {save_path}')
plt.show()


In [ ]:
# 3.3 Distribusi Rating Google Maps
plt.figure(figsize=(8, 4))
sns.histplot(df_umkm['rating'].dropna(), bins=15, kde=True, color='teal')
plt.title('Distribusi Rating Google Maps UMKM Kuliner Surabaya Timur', fontsize=13, fontweight='bold')
plt.xlabel('Rating (1.0 - 5.0)', fontsize=11)
plt.ylabel('Frekuensi', fontsize=11)
plt.tight_layout()
save_path = OUTPUT_DIR / '02_distribusi_rating.png'
plt.savefig(save_path, dpi=300, bbox_inches='tight')
print(f'-> [TERSIMPAN DI GDRIVE] {save_path}')
plt.show()

print('Statistik Deskriptif Rating:')
print(df_umkm['rating'].describe())


---
## 4. EDA Modalitas 2 — Data Teks Ulasan Pelanggan
Menganalisis panjang ulasan, sentimen awal, dan frekuensi penyebutan kata kunci pembayaran digital (QRIS, POS, EDC, Merchant Online).


In [ ]:
# 4.1 Frekuensi Sinyal Digital dalam Ulasan
sinyal_cols = ['mention_qris', 'mention_pos', 'mention_edc', 'mention_online']
sinyal_sum = df_ulasan[sinyal_cols].sum()

plt.figure(figsize=(8, 4))
labels = ['QRIS / Barcode', 'POS / Kasir', 'EDC / Debit', 'Merchant Online (Ojol)']
sns.barplot(x=labels, y=sinyal_sum.values, palette='magma')
plt.title('Frekuensi Penyebutan Indikator Digital dalam Ulasan Pelanggan', fontsize=13, fontweight='bold')
plt.ylabel('Jumlah Kemunculan dalam Ulasan', fontsize=11)
for i, v in enumerate(sinyal_sum.values):
    plt.text(i, v + 1, str(v), ha='center', fontweight='bold')
plt.tight_layout()
save_path = OUTPUT_DIR / '03_frekuensi_indikator_ulasan.png'
plt.savefig(save_path, dpi=300, bbox_inches='tight')
print(f'-> [TERSIMPAN DI GDRIVE] {save_path}')
plt.show()


In [ ]:
# 4.2 Analisis Panjang Kata Ulasan
plt.figure(figsize=(8, 4))
sns.boxplot(x=df_ulasan['kecamatan'], y=df_ulasan['word_count'], palette='Set2')
plt.title('Panjang Kata Ulasan Pelanggan per Kecamatan', fontsize=13, fontweight='bold')
plt.xlabel('Kecamatan', fontsize=11)
plt.ylabel('Jumlah Kata', fontsize=11)
plt.xticks(rotation=20)
plt.tight_layout()
save_path = OUTPUT_DIR / '04_panjang_kata_ulasan.png'
plt.savefig(save_path, dpi=300, bbox_inches='tight')
print(f'-> [TERSIMPAN DI GDRIVE] {save_path}')
plt.show()


---
## 5. EDA Modalitas 3 — Citra Foto Bersih & Indikator Target Klasifikasi (YOLOv8)
Sesuai arsitektur sistem, citra bersih **hanya memuat stan dan indikator digital (QRIS, EDC, POS, Merchant Online)**. Foto makanan murni, minuman, dan menu telah disaring keluar menggunakan detektor objek YOLOv8.


In [ ]:
# 5.1 Distribusi Indikator Target Klasifikasi pada Citra Bersih
ind_cols = ['has_stan', 'has_qris', 'has_pos', 'has_edc', 'has_merchant_online']
ind_labels = ['Stan / Etalase', 'QRIS', 'POS / Kasir', 'Mesin EDC', 'Merchant Online']
ind_counts = [df_foto[col].sum() if col in df_foto.columns else 0 for col in ind_cols]

plt.figure(figsize=(9, 4.5))
sns.barplot(x=ind_labels, y=ind_counts, palette='crest')
plt.title('Distribusi Indikator Target Klasifikasi Terdeteksi pada Citra Bersih (YOLOv8)', fontsize=13, fontweight='bold')
plt.ylabel('Jumlah Foto Memuat Indikator', fontsize=11)
for i, v in enumerate(ind_counts):
    plt.text(i, v + 1, str(int(v)), ha='center', fontweight='bold')
plt.tight_layout()
save_path = OUTPUT_DIR / '05_distribusi_indikator_citra_yolo.png'
plt.savefig(save_path, dpi=300, bbox_inches='tight')
print(f'-> [TERSIMPAN DI GDRIVE] {save_path}')
plt.show()

print('Rincian Keberadaan Indikator pada Citra:')
for lbl, cnt in zip(ind_labels, ind_counts):
    print(f'  - {lbl}: {int(cnt)} foto ({cnt/len(df_foto)*100:.1f}%)')


In [ ]:
# 5.2 Distribusi Kelas Ground Truth Survei Lapangan SWK
df_survei_foto = df_foto[df_foto['sumber'] == 'survei_lapangan_swk']
class_counts = df_survei_foto['label_ground_truth'].value_counts()

plt.figure(figsize=(7, 5))
colors = ['#ff9999', '#66b3ff', '#99ff99']
plt.pie(class_counts.values, labels=class_counts.index, autopct='%1.1f%%', colors=colors, startangle=140, explode=(0.05, 0.05, 0.05))
plt.title('Distribusi Kelas Ground Truth Survei SWK\n(Convention Hall & Deles)', fontsize=13, fontweight='bold')
plt.tight_layout()
save_path = OUTPUT_DIR / '06_distribusi_ground_truth_swk.png'
plt.savefig(save_path, dpi=300, bbox_inches='tight')
print(f'-> [TERSIMPAN DI GDRIVE] {save_path}')
plt.show()

print('Rincian Foto Survei Lapangan:')
for k, v in class_counts.items():
    print(f'  - Kelas {k}: {v} foto')


In [ ]:
# 5.3 Status Anonimisasi Privasi (Blur Wajah)
total_blurred = df_foto['face_blurred_count'].sum()
photos_with_faces = len(df_foto[df_foto['face_blurred_count'] > 0])

print(f'Total wajah pengunjung/kasir yang terdeteksi & diburamkan: {total_blurred} wajah')
print(f'Jumlah foto yang memuat wajah dan telah disanitasi: {photos_with_faces} foto')
print(f'Total foto bersih target yang tersimpan: {len(df_foto)} foto')


---
## 6. EDA Modalitas 4 — Dokumen Regulasi & Perizinan
Mengevaluasi dokumen perizinan penelitian dari instansi terkait.


In [ ]:
# 6.1 Ringkasan Dokumen
display(df_dokumen[['dok_id', 'file_name', 'pages', 'file_size_kb', 'institusi']])

print('\nContoh Ringkasan Teks Dokumen:')
print(df_dokumen.loc[0, 'ringkasan_teks'][:350] + '...')


---
## 7. Kesimpulan & Temuan Kunci EDA
1. **Representasi Spasial:** 6 kecamatan di Surabaya Timur terpetakan dengan baik (total 190 UMKM reguler + stan sentra kuliner).
2. **Kurasi Citra Bersih (YOLOv8):** Dataset foto bersih telah tersaring secara ketat sehingga hanya memuat stan/etalase serta target indikator (QRIS, EDC, POS, Merchant Online). Foto makanan murni telah dieliminasi.
3. **Pola Pembayaran Digital:** QRIS dan Merchant Online merupakan sinyal digital yang paling dominan muncul pada ulasan pelanggan dan citra stan.
4. **Kualitas Ground Truth SWK:** Data survei offline dari SWK Convention Hall dan SWK Deles menyediakan foto stan teranotasi dengan sebaran kelas yang seimbang (9 Konvensional, 16 Berkembang, 19 Maju).
5. **Kepatuhan Privasi:** 100% ulasan teks telah disanitasi dari nomor telepon/identitas pengulas, dan seluruh wajah pada foto telah diburamkan menggunakan OpenCV Gaussian Blur sebelum masuk ke tahap modeling.
6. **Integritas Cloud Storage:** Seluruh artefak visual berhasil diekspor langsung ke folder Google Drive kelompok.
